In [ ]:
"""
FastF1 bulk data pull: telemetry + lap + weather data across seasons.

Run this as a script (not notebook) so it can be safely re-run/resumed
if it dies partway through a multi-hour pull.

Usage:
    python fastf1_pull.py
"""

import time
import socket
import logging
from pathlib import Path

import fastf1
import fastf1.ergast.interface
import pandas as pd
from tqdm import tqdm

socket.setdefaulttimeout(60)

fastf1.ergast.interface.HEADERS["User-Agent"] = (
    f"f1-laptime-prediction/1.0 FastF1/{fastf1.__version__} "
    + fastf1.ergast.interface.HEADERS["User-Agent"]
)

SEASONS = [2023, 2024, 2025]
SESSIONS_TO_PULL = ["R", "Q"]
EXCLUDE_SPRINT_WEEKENDS = False
ONLY_ROUNDS = None
CACHE_DIR = Path("cache")
OUTPUT_DIR = Path("data/raw/quali")
MAX_RETRIES = 3
RETRY_BACKOFF_SECONDS = 15

TELEMETRY_COLUMNS = [
    "Date", "SessionTime", "Distance", "Speed", "Throttle",
    "Brake", "nGear", "RPM", "DRS",
]  
LAP_COLUMNS = [
    "Driver", "Team", "LapNumber", "LapTime", "Stint",
    "Compound", "TyreLife", "FreshTyre", "TrackStatus",
    "IsPersonalBest", "Deleted",
]
WEATHER_COLUMNS = [
    "AirTemp", "TrackTemp", "Humidity", "Pressure",
    "WindSpeed", "WindDirection", "Rainfall",
]

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
)
logger = logging.getLogger("fastf1_pull")


def setup():
    CACHE_DIR.mkdir(exist_ok=True)
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    fastf1.Cache.enable_cache(str(CACHE_DIR))


def get_events_for_season(year: int) -> pd.DataFrame:
    schedule = fastf1.get_event_schedule(year)
    schedule = schedule[schedule["EventFormat"] != "testing"]
    if EXCLUDE_SPRINT_WEEKENDS:
        schedule = schedule[schedule["EventFormat"] == "conventional"]
    if ONLY_ROUNDS is not None:
        schedule = schedule[schedule["RoundNumber"].isin(ONLY_ROUNDS)]
    return schedule


def load_session_with_retry(year: int, round_number: int, session_code: str):
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            session = fastf1.get_session(year, round_number, session_code)
            session.load(telemetry=True, laps=True, weather=True, messages=False)

            if session.laps is None or session.laps.empty:
                raise RuntimeError("session.load() completed but laps data is empty")

            try:
                session.load(telemetry=False, laps=False, weather=False, messages=True)
            except Exception as e:
                logger.warning(
                    f"[{year} R{round_number} {session_code}] race control messages "
                    f"failed to load, continuing without them: {e}"
                )

            return session
        except Exception as e:
            logger.warning(
                f"[{year} R{round_number} {session_code}] attempt {attempt}/{MAX_RETRIES} "
                f"failed: {e}"
            )
            if attempt < MAX_RETRIES:
                time.sleep(RETRY_BACKOFF_SECONDS * attempt)
            else:
                logger.error(
                    f"[{year} R{round_number} {session_code}] giving up after {MAX_RETRIES} attempts"
                )
                return None


def extract_session_dataframe(session) -> pd.DataFrame | None:
    """
    Bulk-pull telemetry per driver (one call per driver, not per lap), then
    attach lap metadata and weather via merge_asof. Far fewer, larger
    operations than a per-lap loop -- meaningfully faster at this scale.

    Works unchanged for qualifying: FastF1 exposes the same session.laps /
    session.car_data / session.weather_data shape for Q as for R. Qualifying
    laps include in- and out-laps and banned/deleted laps same as race laps;
    those get filtered downstream in build_features.py the same way, plus
    the eventual Q-specific feature step will need to select each driver's
    single fastest clean lap out of this raw pull rather than treat every
    row as a lap-time target the way race laps are.
    """
    laps = session.laps
    if laps is None or laps.empty:
        return None

    if "LapStartDate" not in laps.columns:
        raise RuntimeError(
            "session.laps is missing 'LapStartDate' -- cannot align lap metadata "
            "to absolute telemetry timestamps. Check your fastf1 version."
        )

    laps_meta = laps[[c for c in LAP_COLUMNS if c in laps.columns] + ["LapStartDate", "DriverNumber"]].copy()
    laps_meta = laps_meta.sort_values("LapStartDate")

    weather = session.weather_data.copy() if session.weather_data is not None else None
    if weather is not None and not weather.empty:
        weather["Date"] = session.t0_date + weather["Time"]
        weather = weather.sort_values("Date")

    all_drivers = []
    for drv in session.drivers:
        try:
            car_data = session.car_data[drv].add_distance()
        except Exception as e:
            logger.warning(f"Skipping driver {drv}: no car data ({e})")
            continue

        tel = car_data[[c for c in TELEMETRY_COLUMNS if c in car_data.columns]].copy()
        tel["DriverNumber"] = str(drv)
        tel = tel.sort_values("Date") if "Date" in tel.columns else tel

        driver_laps = laps_meta[laps_meta["DriverNumber"] == str(drv)]

        if not driver_laps.empty and "Date" in tel.columns:
            tel = pd.merge_asof(
                tel,
                driver_laps.rename(columns={"LapStartDate": "Date"}),
                on="Date",
                direction="backward",
                tolerance=pd.Timedelta(minutes=2),
            )

        if weather is not None and not weather.empty and "Date" in tel.columns:
            tel = pd.merge_asof(
                tel.sort_values("Date"),
                weather[["Date"] + [c for c in WEATHER_COLUMNS if c in weather.columns]],
                on="Date",
                direction="nearest",
            )

        all_drivers.append(tel)

    if not all_drivers:
        return None

    result = pd.concat(all_drivers, ignore_index=True)
    result["Year"] = session.event["EventDate"].year
    result["RoundNumber"] = session.event["RoundNumber"]
    result["EventName"] = session.event["EventName"]
    result["SessionType"] = session.name
    result["EventFormat"] = session.event.get("EventFormat", None)  
    return result


def extract_race_control(session) -> pd.DataFrame | None:
    """
    Race control messages (flags, Safety Car, VSC, penalties, etc).
    Kept as a separate file rather than merged into telemetry, since these
    are sparse events, not per-timestamp readings -- join them onto
    telemetry/laps downstream by timestamp when you build features
    (e.g. 'was a Safety Car active during this lap').

    Qualifying sessions can also throw red flags (e.g. a crash in Q1/Q2/Q3),
    so this is pulled for Q the same as for R -- build_features.py's
    red-flag logic doesn't currently run on Q data, but the raw file is
    saved regardless in case that's needed later.
    """
    rcm = session.race_control_messages
    if rcm is None or rcm.empty:
        return None
    rcm = rcm.copy()
    rcm["Year"] = session.event["EventDate"].year
    rcm["RoundNumber"] = session.event["RoundNumber"]
    rcm["EventName"] = session.event["EventName"]
    return rcm


def output_path(year: int, round_number: int, session_code: str) -> Path:
    return (
        OUTPUT_DIR
        / f"year={year}"
        / f"round={round_number}"
        / f"{session_code}.parquet"
    )


def race_control_output_path(year: int, round_number: int, session_code: str) -> Path:
    return (
        OUTPUT_DIR
        / f"year={year}"
        / f"round={round_number}"
        / f"{session_code}_race_control.parquet"
    )


def main():
    setup()

    for year in SEASONS:
        events = get_events_for_season(year)
        logger.info(f"Season {year}: {len(events)} events to pull")

        for _, event in tqdm(events.iterrows(), total=len(events), desc=f"{year}"):
            round_number = event["RoundNumber"]

            for session_code in SESSIONS_TO_PULL:
                out_path = output_path(year, round_number, session_code)

                if out_path.exists():
                    logger.info(f"Skipping {out_path} (already exists)")
                    continue

                try:
                    t0 = time.perf_counter()
                    session = load_session_with_retry(year, round_number, session_code)
                    load_time = time.perf_counter() - t0
                    if session is None:
                        continue

                    t1 = time.perf_counter()
                    df = extract_session_dataframe(session)
                    extract_time = time.perf_counter() - t1

                    logger.info(
                        f"[{year} R{round_number} {session_code}] "
                        f"load={load_time:.1f}s extract={extract_time:.1f}s"
                    )

                    if df is None or df.empty:
                        logger.warning(
                            f"No data extracted for {year} R{round_number} {session_code}"
                        )
                        continue

                    out_path.parent.mkdir(parents=True, exist_ok=True)
                    df.to_parquet(out_path, index=False)
                    logger.info(f"Saved {out_path} ({len(df):,} rows)")

                    rc_df = extract_race_control(session)
                    if rc_df is not None and not rc_df.empty:
                        rc_path = race_control_output_path(year, round_number, session_code)
                        rc_df.to_parquet(rc_path, index=False)
                        logger.info(f"Saved {rc_path} ({len(rc_df):,} rows)")

                except Exception as e:
                    logger.exception(
                        f"[{year} R{round_number} {session_code}] unexpected failure, "
                        f"skipping this session"
                    )
                    continue

    logger.info("Pull complete.")


if __name__ == "__main__":
    main()